In [14]:
%pip install --upgrade pip
%pip uninstall cryptography -y
%pip install cryptography==41.0.7
%pip install google-generativeai python-dotenv


Found existing installation: cryptography 41.0.7Note: you may need to restart the kernel to use updated packages.

Uninstalling cryptography-41.0.7:
  Successfully uninstalled cryptography-41.0.7


You can safely remove it manually.


  Using cached cryptography-41.0.7-cp37-abi3-win_amd64.whl.metadata (5.3 kB)
Using cached cryptography-41.0.7-cp37-abi3-win_amd64.whl (2.7 MB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [15]:
import google.generativeai as genai
print("Fixed! Working")

Fixed! Working


In [16]:
%pip install --upgrade pip
%pip install -q -U google-generativeai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import numpy as np
import google.generativeai as genai
import os

print(genai.__version__) # should show version

# check attribute
print(dir(genai)[:20])

0.8.6
['ChatSession', 'GenerationConfig', 'GenerativeModel', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'annotations', 'caching', 'configure', 'create_tuned_model', 'delete_file', 'delete_tuned_model', 'embed_content']


In [18]:
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-3.6-flash")

print("Gemini model loaded successfully!")

Gemini model loaded successfully!


In [19]:
file_path = "../data/House Price India.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print(df.shape)

Dataset loaded successfully!
(14619, 23)


In [20]:
for col in df.columns:

    if df[col].isnull().sum() > 0:

        # Object / categorical column
        if df[col].dtype == "object":

            df[col].fillna(
                df[col].mode()[0],
                inplace=True
            )

        # Numerical column
        else:

            if abs(df[col].skew()) >= 1:

                # Skewed → Median
                df[col].fillna(
                    df[col].median(),
                    inplace=True
                )

            else:

                # Approximately normal → Mean
                df[col].fillna(
                    df[col].mean(),
                    inplace=True
                )

print("Missing values handled!")

Missing values handled!


In [21]:
summary = f"""
Dataset Shape:
{df.shape}

Column Names:
{list(df.columns)}

Data Types:
{df.dtypes.to_string()}

Missing Values:
{df.isnull().sum().to_string()}

Numerical Summary:
{df.describe(include="number").to_string()}

Categorical Columns:
{list(df.select_dtypes(include="object").columns)}
"""

In [22]:
prompt = f"""
You are an expert Data Analyst.

Analyze the following dataset and provide a clear,
simple and professional analysis.

Provide the following sections:

1. Dataset Overview
- Explain what the dataset contains.
- Mention number of rows and columns.
- Explain important columns.

2. Data Quality
- Mention missing values.
- Mention possible data quality issues.
- Mention unusual values or possible outliers if visible.

3. Important Trends
- Identify important patterns in the numerical data.
- Explain important categorical distributions.

4. Feature Relationships
- Explain possible relationships between important features.
- Mention which features may be useful for prediction.

5. Business Insights
- Give practical insights based on the dataset.

6. Recommendations
- Give useful recommendations based on the analysis.

Do NOT invent information that is not supported by the dataset.

Dataset Summary:

{summary}
"""

In [23]:
response = model.generate_content(prompt)

print(response.text)

Here is a professional and structured analysis of the provided dataset.

---

### 1. Dataset Overview

* **Content**: The dataset contains real estate property records detailing housing characteristics, locations, structural dimensions, condition ratings, and selling prices.
* **Dataset Size**: 14,619 rows and 23 columns.
* **Key Columns**:
  * **Target Feature**: `Price` (the sale price of the house).
  * **Size & Structure**: `living area`, `lot area`, `number of bedrooms`, `number of bathrooms`, `number of floors`, `Area of the house(excluding basement)`, `Area of the basement`.
  * **Quality & Features**: `grade of the house` (1–13 scale), `condition of the house` (1–5 scale), `waterfront present` (binary indicator), `number of views`.
  * **Location**: `Postal Code`, `Lattitude`, `Longitude`, `Distance from the airport`, `Number of schools nearby`.
  * **Time/Age**: `Built Year`, `Renovation Year`, `Date`.

---

### 2. Data Quality

* **Missing Values**: There are **0 missing valu